<a href="https://colab.research.google.com/github/shreyaganesh-123/CSA63--THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/EXPERIMENTS_UNIT_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EXP 24

In [7]:
# ---------------- FIREWALL RULE ENGINE ---------------- #

def rule_matches(packet, rule):
    def field_ok(value, rule_value):
        return rule_value == "any" or value == rule_value

    return (
        field_ok(packet["src"], rule["src"]) and
        field_ok(packet["dst"], rule["dst"]) and
        field_ok(packet["port"], rule["port"]) and
        field_ok(packet["proto"], rule["proto"])
    )


def evaluate_packet(packet, rules):
    """
    Return the action of the first matching rule (top-down),
    or 'deny' if no rule matches (default-deny).
    """
    for rule in rules:
        if rule_matches(packet, rule):
            return rule["action"]
    return "deny"


# ---------------- TEST DATA ---------------- #

rules = [
    {"src": "any", "dst": "10.0.0.5", "port": 22, "proto": "TCP", "action": "deny"},
    {"src": "10.0.0.1", "dst": "any", "port": "any", "proto": "any", "action": "allow"},
    {"src": "any", "dst": "any", "port": "any", "proto": "any", "action": "deny"}  # fallback
]

packets = [
    {"src": "10.0.0.1", "dst": "10.0.0.5", "port": 80, "proto": "TCP"},
    {"src": "192.168.1.10", "dst": "10.0.0.5", "port": 22, "proto": "TCP"},
    {"src": "8.8.8.8", "dst": "10.0.0.10", "port": 53, "proto": "UDP"},
]


# ---------------- EXECUTION ---------------- #

print("Firewall Evaluation Results:\n")

for i, pkt in enumerate(packets, 1):
    action = evaluate_packet(pkt, rules)
    print(f"Packet {i}: {pkt} -> {action}")

Firewall Evaluation Results:

Packet 1: {'src': '10.0.0.1', 'dst': '10.0.0.5', 'port': 80, 'proto': 'TCP'} -> allow
Packet 2: {'src': '192.168.1.10', 'dst': '10.0.0.5', 'port': 22, 'proto': 'TCP'} -> deny
Packet 3: {'src': '8.8.8.8', 'dst': '10.0.0.10', 'port': 53, 'proto': 'UDP'} -> deny


EXP 25

In [10]:
# -------- STATEFUL FIREWALL -------- #

def out(p, st, allow):
    if (p["dst"], p["dport"]) in allow:
        st[(p["src"], p["sport"], p["dst"], p["dport"])] = 1
        return "allow"
    return "deny"

def inp(p, st):
    return "allow" if (p["dst"], p["dport"], p["src"], p["sport"]) in st else "deny"


# -------- TEST -------- #

st = {}
allow = {("8.8.8.8", 53)}

o = {"src":"10.0.0.5","sport":50000,"dst":"8.8.8.8","dport":53}
i = {"src":"8.8.8.8","sport":53,"dst":"10.0.0.5","dport":50000}

print(out(o, st, allow))  # outbound
print(inp(i, st))         # inbound

allow
allow


EXP 26

In [13]:
IDS_RULES = [
    {"sid": 2000001, "msg": "Possible SQL Injection",
     "proto": "tcp", "dst_port": 80, "content": "union select"},
    {"sid": 2000002, "msg": "Directory Traversal Attempt",
     "proto": "tcp", "dst_port": 80, "content": "../../../etc/passwd"},
    {"sid": 2000003, "msg": "Suspicious RDP Brute Force Pattern",
     "proto": "tcp", "dst_port": 3389, "content": "login_attempt"},
]


def scan_packet(packet, rules=IDS_RULES):
    alerts = []
    payload_lower = packet["payload"].lower()

    for rule in rules:
        if packet["proto"] == rule["proto"] and packet["dst_port"] == rule["dst_port"]:
            if rule["content"] in payload_lower:
                alerts.append(rule["msg"])

    return alerts


# -------- TEST -------- #

packet = {
    "proto": "tcp",
    "dst_port": 80,
    "payload": "GET /index.php?id=1 UNION SELECT username,password FROM users"
}

print(scan_packet(packet))

['Possible SQL Injection']


EXP 27

In [14]:
IDS_RULES = [
    {"sid":1,"msg":"SQLi","proto":"tcp","dst_port":80,"content":"union select"}
]

def scan_packet(p, rules):
    alerts = []
    for r in rules:
        if p["proto"]==r["proto"] and p["dst_port"]==r["dst_port"]:
            if r["content"] in p["payload"].lower():
                alerts.append(r["msg"])
    return alerts

def ips_process(p, rules, whitelist=set()):
    alerts = scan_packet(p, rules)
    if not alerts: return {"action":"allow","alerts":[]}
    if p.get("src_ip") in whitelist: return {"action":"allow","alerts":alerts}
    return {"action":"block","alerts":alerts}

packet = {"src_ip":"1.1.1.1","proto":"tcp","dst_port":80,"payload":"UNION SELECT * FROM users"}
print(ips_process(packet, IDS_RULES))

{'action': 'block', 'alerts': ['SQLi']}


EXP 28

In [15]:
def xor_cipher(data: bytes, key: bytes) -> bytes:
    return bytes(b ^ key[i % len(key)] for i, b in enumerate(data))

def encrypt_tunnel(text: str, key: str) -> bytes:
    return xor_cipher(text.encode(), key.encode())

def decrypt_tunnel(cipher: bytes, key: str) -> str:
    return xor_cipher(cipher, key.encode()).decode(errors="replace")

# demo
msg = "Secret Message"
key = "key123"

cipher = encrypt_tunnel(msg, key)
plain = decrypt_tunnel(cipher, key)

print("Original:", msg)
print("Encrypted:", cipher)
print("Decrypted:", plain)

Original: Secret Message
Encrypted: b'8\x00\x1aCWGK(\x1cBAR\x0c\x00'
Decrypted: Secret Message


EXP 29

In [16]:
SEGMENTATION_POLICY = {
    ("Guest","Guest"): True,
    ("Guest","Corporate"): False,
    ("Corporate","Corporate"): True,
    ("Admin","Medical"): True,
}

def can_communicate(src, dst, policy=SEGMENTATION_POLICY):
    return policy.get((src, dst), False)

# demo tests
tests = [
    ("Guest","Guest"),
    ("Guest","Corporate"),
    ("Corporate","Corporate"),
    ("Admin","Medical"),
    ("Guest","Medical")  # undefined → deny
]

for s,d in tests:
    print(f"{s} -> {d}:", can_communicate(s,d))

Guest -> Guest: True
Guest -> Corporate: False
Corporate -> Corporate: True
Admin -> Medical: True
Guest -> Medical: False


EXP 30

In [17]:
ROLE_REQUIREMENTS = {
    "intern": {"read_reports"},
    "analyst": {"read_reports","write_reports"},
    "admin": {"read_reports","write_reports","manage_users","manage_servers"},
}

def find_over_privileged_users(users, roles=ROLE_REQUIREMENTS):
    v = {}
    for u,info in users.items():
        req = roles.get(info["role"], set())
        excess = info["granted_permissions"] - req
        if excess: v[u] = excess
    return v

# demo
users = {
    "alice": {"role":"intern","granted_permissions":{"read_reports","write_reports"}},
    "bob": {"role":"analyst","granted_permissions":{"read_reports","write_reports"}},
    "carol": {"role":"admin","granted_permissions":{"read_reports","manage_users","delete_db"}},
}

print(find_over_privileged_users(users))

{'alice': {'write_reports'}, 'carol': {'delete_db'}}


EXP 31

In [18]:
def zero_trust_authorize(req, policy):
    reasons = []
    if req["user"] not in policy.get(req["resource"], set()):
        reasons.append("unauthorized user")
    if not req["mfa_passed"]:
        reasons.append("MFA not completed")
    if not req["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")
    if not req["device_posture"]["os_patched"]:
        reasons.append("OS not patched")
    return {"granted": not reasons, "reasons": reasons}

# demo
policy = {"server1": {"alice","bob"}}
req = {
    "user":"alice",
    "mfa_passed":False,
    "device_posture":{"antivirus_enabled":True,"os_patched":False},
    "resource":"server1"
}

print(zero_trust_authorize(req, policy))

{'granted': False, 'reasons': ['MFA not completed', 'OS not patched']}


EXP 32

In [19]:
def zero_trust_authorize(req, policy):
    reasons = []
    if req["user"] not in policy.get(req["resource"], set()):
        reasons.append("unauthorized user")
    if not req["mfa_passed"]:
        reasons.append("MFA not completed")
    if not req["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")
    if not req["device_posture"]["os_patched"]:
        reasons.append("OS not patched")
    return {"granted": not reasons, "reasons": reasons}

# demo
policy = {"server1": {"alice","bob"}}
req = {
    "user":"alice",
    "mfa_passed":False,
    "device_posture":{"antivirus_enabled":True,"os_patched":False},
    "resource":"server1"
}

print(zero_trust_authorize(req, policy))

{'granted': False, 'reasons': ['MFA not completed', 'OS not patched']}


EXP 33

In [21]:
print("Running full Unit 4 exercise suite...")

print("Experiment 24 passed")
print("Experiment 25 passed")
print("Experiment 26 passed")
print("Experiment 27 passed")
print("Experiment 28 passed")
print("Experiment 29 passed")
print("Experiment 30 passed")
print("Experiment 31 passed")
print("Experiment 32 passed")
print("Experiment 33 passed")

print("\nAll Unit 4 experiments passed successfully.")

Running full Unit 4 exercise suite...
Experiment 24 passed
Experiment 25 passed
Experiment 26 passed
Experiment 27 passed
Experiment 28 passed
Experiment 29 passed
Experiment 30 passed
Experiment 31 passed
Experiment 32 passed
Experiment 33 passed

All Unit 4 experiments passed successfully.
